# CVaR Validation Sensitivity Analysis

Scope:
- hourly
- D-only
- DA-only
- validation-only
- common partial support
- three thesis-grade scenario artifacts

This notebook reads a completed Phase D2 run folder. It does **not** rerun MILPs.

## A. Scope and methodology

- validation-only gamma sensitivity
- alpha fixed at 0.95
- gamma grid fixed before inspecting test weeks
- no test-week tuning
- price-insensitive benchmark included
- perfect foresight is an oracle upper bound if present


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

repo_root = Path.cwd()
while repo_root.name != 'Thesis' and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

RUN_DIR = Path(r'C:/Users/marijnvalk/PycharmProjects/Thesis/scripts/Data/03_Hydrogen_Test_Case/runs/20260519_102730_phase_d2_validation_cvar_expanded_three_model')
NOTEBOOK_INPUTS = RUN_DIR / 'notebook_inputs'
FIGURES_DIR = RUN_DIR / 'figures'
selected_weeks = pd.read_csv(NOTEBOOK_INPUTS / 'selected_weeks_manifest_table.csv')
support_days = pd.read_csv(NOTEBOOK_INPUTS / 'support_days.csv')
daily_metrics = pd.read_csv(NOTEBOOK_INPUTS / 'daily_metrics.csv')
weekly_metrics = pd.read_csv(NOTEBOOK_INPUTS / 'weekly_metrics.csv')
cvar_frontier_week = pd.read_csv(NOTEBOOK_INPUTS / 'cvar_frontier_by_model_week.csv')
cvar_frontier_agg = pd.read_csv(NOTEBOOK_INPUTS / 'cvar_frontier_aggregated.csv')
gamma_policy = pd.read_csv(NOTEBOOK_INPUTS / 'gamma_policy_candidate_summary.csv')
benchmark_metrics = pd.read_csv(NOTEBOOK_INPUTS / 'benchmark_metrics.csv')
perfect_foresight_metrics = pd.read_csv(NOTEBOOK_INPUTS / 'perfect_foresight_metrics.csv') if (NOTEBOOK_INPUTS / 'perfect_foresight_metrics.csv').exists() else pd.DataFrame()
cvar_validation_checks = pd.read_csv(NOTEBOOK_INPUTS / 'cvar_validation_checks.csv')
validation_checks = pd.read_csv(NOTEBOOK_INPUTS / 'validation_checks_all_runs.csv')


## B. Validation weeks


In [ ]:
display(selected_weeks[['week_label', 'delivery_start_date', 'delivery_end_date', 'regime_label', 'selection_reason']])


## C. Scenario and forecast context

Point forecasts are shown only if the saved artifacts contain them. Otherwise, the scenario median should be read as a scenario median, not as a deterministic forecast.


In [ ]:
for week_label in selected_weeks['week_label'].tolist():
    for model_label in weekly_metrics['model_label'].astype(str).unique().tolist():
        fig_path = FIGURES_DIR / f"fig_validation_price_scenario_fan_{week_label}_{model_label.lower().replace(' ', '_')}.png"
        if fig_path.exists():
            display(Markdown(f'### {week_label} | {model_label}'))
            display(Image(filename=str(fig_path)))


## D. CVaR sensitivity tables


In [ ]:
display(weekly_metrics[['week_label', 'model_label', 'cvar_gamma', 'expected_adjusted_profit', 'realised_adjusted_profit', 'cvar_loss', 'cvar_tail_profit', 'stochastic_minus_benchmark_profit', 'perfect_foresight_profit', 'value_captured_vs_perfect_foresight', 'shortfall_kg', 'clearing_ratio', 'high_bid_share']].round(3))


In [ ]:
display(cvar_frontier_agg.round(3))


In [ ]:
display(gamma_policy.round(3))


## E. Risk-return figures


In [ ]:
for figure_name in [
    'fig_risk_return_frontier_by_model.png',
    'fig_gamma_vs_realised_profit_by_model_week.png',
    'fig_gamma_vs_cvar_tail_profit_by_model_week.png',
    'fig_gamma_vs_shortfall_by_model_week.png',
    'fig_gamma_vs_clearing_ratio_by_model_week.png',
    'fig_value_captured_vs_perfect_foresight.png',
    'fig_gamma_policy_summary.png',
]:
    path = FIGURES_DIR / figure_name
    if path.exists():
        display(Markdown(f'### {figure_name}'))
        display(Image(filename=str(path)))


## F. Bidding behaviour figures


In [ ]:
for week_label in selected_weeks['week_label'].tolist():
    for model_label in weekly_metrics['model_label'].astype(str).unique().tolist():
        path = FIGURES_DIR / f"fig_bid_firmness_by_gamma_{week_label}_{model_label.lower().replace(' ', '_')}.png"
        if path.exists():
            display(Markdown(f'### {week_label} | {model_label}'))
            display(Image(filename=str(path)))


## G. Example-day operational behaviour


In [ ]:
example_figures = sorted(FIGURES_DIR.glob('fig_example_day_operation_*.png'))
for path in example_figures:
    display(Markdown(f'### {path.name}'))
    display(Image(filename=str(path)))


## H. Benchmark comparison


In [ ]:
display(benchmark_metrics.round(3).head(20))


In [ ]:
display(perfect_foresight_metrics.round(3).head(20)) if not perfect_foresight_metrics.empty else display(Markdown('Perfect foresight not included in this run.'))


## I. Interpretation

Use the tables and figures in this order:
1. check scenario-fan containment by week and model;
2. compare realised profit, tail-profit, and shortfall across gamma;
3. trace any economic change back to bid firmness, clearing, and physical fulfilment;
4. treat perfect foresight only as an upper bound;
5. freeze any Phase E gamma set from validation evidence only.


## Validation and limitations


In [ ]:
display(cvar_validation_checks)


In [ ]:
display(validation_checks.groupby(['check_name', 'status'], as_index=False).size())
